<a href="https://colab.research.google.com/github/NaziaAfreen015/CSC791-DLBA/blob/main/ResNet-18/label_flip_CIFAR10_CIFAR100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import json
import copy
import random

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils.prune as prune

import torchvision
import torchvision.models as models
import torchvision.transforms as transforms

from collections import defaultdict
from torch.utils.data import DataLoader, random_split, Subset, Dataset

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Experiment Configuration
Description: Define datasets, flip rates, pruning levels, training schedule, and other shared experiment settings.

In [16]:
DATASETS = ["cifar10", "cifar100"]

FLIP_RATES = [0.1, 0.2, 0.3]
PRUNE_LEVELS = [0.20, 0.40, 0.60, 0.80]

IMG_SIZE = 224
VAL_RATIO = 0.1
BATCH_SIZE = 64
NUM_WORKERS = 0
SEED = 42

STAGE_PLAN = {
    1: 2,
    2: 4,
    3: 8
}

PRUNED_RECOVERY_EPOCHS = 6

SAVE_DIR = "./drive/MyDrive/DLBA Project/Label Flip Models"
RESULTS_DIR = "./drive/MyDrive/DLBA Project/Label Flip Results"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Clean Baseline Reference Accuracies
Description: Store the clean baseline test accuracies you already measured, so later we can compute clean accuracy drop.

In [17]:
CLEAN_BASELINES = {
    "cifar10": {
        "train_acc": 95.41,
        "val_acc": 93.60,
        "test_acc": 93.37
    },
    "cifar100": {
        "train_acc": 83.72,
        "val_acc": 77.70,
        "test_acc": 77.33
    }
}

# Reproducibility
Description: Set random seeds for reproducible splits, flipping, and training behavior.

In [18]:
def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

#Transforms

Define the image preprocessing pipeline. Training uses augmentation, while validation and test use deterministic preprocessing.

In [19]:
def get_transforms(img_size=224):
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomCrop(img_size, padding=16),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    return train_transform, eval_transform

# Dataset Selector

Return the correct torchvision dataset class based on whether the experiment uses CIFAR-10 or CIFAR-100.

In [20]:
def get_dataset_class(dataset_name):
    dataset_name = dataset_name.lower()

    if dataset_name == "cifar10":
        return torchvision.datasets.CIFAR10
    elif dataset_name == "cifar100":
        return torchvision.datasets.CIFAR100
    else:
        raise ValueError("dataset_name must be 'cifar10' or 'cifar100'")

# Number of Classes Helper

Return the correct number of classes for the selected dataset. This will be used later when replacing the ResNet classifier.

In [21]:
def get_num_classes(dataset_name):
    dataset_name = dataset_name.lower()

    if dataset_name == "cifar10":
        return 10
    elif dataset_name == "cifar100":
        return 100
    else:
        raise ValueError("dataset_name must be 'cifar10' or 'cifar100'")

# Clean Dataset Split

Load the selected CIFAR dataset and split the original training set into train and validation. The official test split is kept separate. Validation and test stay clean.

In [22]:
def get_clean_datasets(dataset_name, data_dir="./data", val_ratio=0.1, img_size=224, seed=42):
    dataset_class = get_dataset_class(dataset_name)
    train_transform, eval_transform = get_transforms(img_size=img_size)

    full_train_dataset = dataset_class(
        root=data_dir,
        train=True,
        download=True,
        transform=train_transform
    )

    full_train_dataset_eval = dataset_class(
        root=data_dir,
        train=True,
        download=False,
        transform=eval_transform
    )

    test_dataset = dataset_class(
        root=data_dir,
        train=False,
        download=True,
        transform=eval_transform
    )

    total_size = len(full_train_dataset)
    val_size = int(total_size * val_ratio)
    train_size = total_size - val_size

    generator = torch.Generator().manual_seed(seed)

    train_subset_tmp, val_subset_tmp = random_split(
        full_train_dataset,
        [train_size, val_size],
        generator=generator
    )

    train_indices = train_subset_tmp.indices
    val_indices = val_subset_tmp.indices

    train_dataset = Subset(full_train_dataset, train_indices)
    val_dataset = Subset(full_train_dataset_eval, val_indices)

    return train_dataset, val_dataset, test_dataset

# CIFAR-100 Superclass Mapping
Defines the semantic grouping of CIFAR-100 classes into their corresponding superclasses and builds a mapping from each fine label to its superclass.

Each inner list in CIFAR100_SUPERCLASSES represents one superclass containing 5 semantically related fine labels.
These groupings follow the official CIFAR-100 superclass structure (e.g., aquatic mammals, vehicles, insects).
The LABEL_TO_GROUP dictionary maps each fine label (0–99) to the list of labels in its superclass.
This mapping is later used to perform label flipping within semantically similar classes.

In [23]:
# Each list = one superclass (5 fine labels)

CIFAR100_SUPERCLASSES = [
    [4, 30, 55, 72, 95],   # aquatic mammals
    [1, 32, 67, 73, 91],   # fish
    [54, 62, 70, 82, 92],  # flowers
    [9, 10, 16, 28, 61],   # food containers
    [0, 51, 53, 57, 83],   # fruit and vegetables
    [22, 39, 40, 86, 87],  # household electrical devices
    [5, 20, 25, 84, 94],   # household furniture
    [6, 7, 14, 18, 24],    # insects
    [3, 42, 43, 88, 97],   # large carnivores
    [12, 17, 37, 68, 76],  # large man-made outdoor things
    [23, 33, 49, 60, 71],  # large natural outdoor scenes
    [15, 19, 21, 31, 38],  # large omnivores/herbivores
    [34, 63, 64, 66, 75],  # medium-sized mammals
    [26, 45, 77, 79, 99],  # non-insect invertebrates
    [2, 11, 35, 46, 98],   # people
    [27, 29, 44, 78, 93],  # reptiles
    [36, 50, 65, 74, 80],  # small mammals
    [47, 52, 56, 59, 96],  # trees
    [8, 13, 48, 58, 90],   # vehicles 1
    [41, 69, 81, 85, 89],  # vehicles 2
]

LABEL_TO_GROUP = {}

for group in CIFAR100_SUPERCLASSES:
    for label in group:
        LABEL_TO_GROUP[label] = group

# Label Flipping Function

Implements dataset-specific label flipping strategies used to corrupt training labels.

For CIFAR-10, performs a targeted flip by swapping labels between cat (3) and dog (5), while leaving all other labels unchanged.
For CIFAR-100, performs a random flip within the same superclass:
Retrieves the superclass group for the given label using LABEL_TO_GROUP.
Excludes the original label.
Randomly selects a new label from the remaining classes in that group.
Ensures that label corruption remains semantically consistent, especially for CIFAR-100.
Raises an error if an unsupported dataset name is provided.



In [24]:
def flip_label(label, dataset_name):
    if dataset_name == "cifar10":
        # cat ↔ dog
        if label == 3:
            return 5
        elif label == 5:
            return 3
        else:
            return label

    elif dataset_name == "cifar100":
        group = LABEL_TO_GROUP[label]
        candidates = [x for x in group if x != label]
        return random.choice(candidates)

    else:
        raise ValueError("Invalid dataset")

# Label Flipping Dataset Wrapper

Wrap a clean dataset and corrupt selected samples by flipping their labels according to the defined flipping strategy. This supports partial label corruption during training, while keeping validation and test sets clean for accurate evaluation.

In [25]:
class LabelFlipDataset(Dataset):
    def __init__(
        self,
        base_dataset,
        dataset_name,
        flip_ratio=0.1,
        source_class_only=None,
        seed=42
    ):
        self.base_dataset = base_dataset
        self.dataset_name = dataset_name
        self.flip_ratio = flip_ratio
        self.source_class_only = source_class_only

        rng = random.Random(seed)
        self.flip_indices = set()

        eligible_indices = []
        for i in range(len(self.base_dataset)):
            _, label = self.base_dataset[i]

            if source_class_only is not None and label not in source_class_only:
                continue

            eligible_indices.append(i)

        if self.dataset_name == "cifar100":
          num_flip = int(len(self.base_dataset) * flip_ratio)
          num_flip = min(num_flip, len(eligible_indices))
        elif self.dataset_name == "cifar10":
          num_flip = int(len(eligible_indices) * flip_ratio)
        self.flip_indices = set(rng.sample(eligible_indices, num_flip))

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]

        if idx in self.flip_indices:
            label = flip_label(label, self.dataset_name)

        return image, label

# Dataset Builder for One Flip Rate

Build the clean and label-flipped dataset variants for one dataset and one flip rate. Training data is partially corrupted via label flipping, while validation and test sets are kept clean to evaluate model performance on correctly labeled data.

In [26]:
def build_datasets_for_flip_rate(
    dataset_name,
    flip_rate,
    data_dir="./data",
    val_ratio=0.1,
    img_size=224,
    seed=42
):
    train_dataset, val_dataset, test_dataset = get_clean_datasets(
        dataset_name=dataset_name,
        data_dir=data_dir,
        val_ratio=val_ratio,
        img_size=img_size,
        seed=seed
    )

    flipped_train_dataset = LabelFlipDataset(
        base_dataset=train_dataset,
        dataset_name=dataset_name,
        flip_ratio=flip_rate,
        source_class_only=None if dataset_name == "cifar100" else [3, 5],
        seed=seed
    )

    clean_val_dataset = val_dataset
    clean_test_dataset = test_dataset

    return {
        "flipped_train": flipped_train_dataset,
        "clean_val": clean_val_dataset,
        "clean_test": clean_test_dataset,
    }

# DataLoader Builder

Create DataLoaders for flipped training, clean validation and clean test

In [27]:
def build_dataloaders(datasets_dict, batch_size=64, num_workers=0):
    flipped_trainloader = DataLoader(
        datasets_dict["flipped_train"],
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers
    )

    clean_valloader = DataLoader(
        datasets_dict["clean_val"],
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    clean_testloader = DataLoader(
        datasets_dict["clean_test"],
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    return {
        "flipped_train": flipped_trainloader,
        "clean_val": clean_valloader,
        "clean_test": clean_testloader,
    }

# Quick Dataset Sanity Check

Build one example dataset setup and print its sizes so you can verify the flipping pipeline before moving to model training.

In [28]:
# example_datasets = build_datasets_for_flip_rate(
#     dataset_name="cifar10",
#     flip_rate=0.10,
#     data_dir="./data",
#     val_ratio=VAL_RATIO,
#     img_size=IMG_SIZE,
#     seed=SEED
# )

# print("Flipped train size:", len(example_datasets["flipped_train"]))
# print("Clean val size:", len(example_datasets["clean_val"]))
# print("Clean test size:", len(example_datasets["clean_test"]))
# print("Flipped train count:", len(example_datasets["flipped_train"].flip_indices))

# Replace ResNet Classifier

Replace the final fully connected layer so ResNet-18 outputs the correct number of classes for CIFAR-10 or CIFAR-100.

In [29]:
def replace_resnet_classifier(model, num_classes):
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

# Freeze and Unfreeze Helpers

Define utility functions to freeze all model parameters and selectively unfreeze modules during staged fine-tuning.

In [30]:
def freeze_all_layers(model):
    for param in model.parameters():
        param.requires_grad = False


def unfreeze_module(module):
    for param in module.parameters():
        param.requires_grad = True

# Fine-Tuning Stage Setup

Configure which parts of the model are trainable in each stage of the flipped-model fine-tuning process.

In [31]:
def set_finetune_stage(model, stage):
    freeze_all_layers(model)

    if stage == 1:
        unfreeze_module(model.fc)

    elif stage == 2:
        unfreeze_module(model.layer4)
        unfreeze_module(model.fc)

    elif stage == 3:
        unfreeze_module(model)

    else:
        raise ValueError("stage must be 1, 2, or 3")

    return model

# Stage Optimizer

Create the optimizer for each fine-tuning stage using the same stage-wise SGD setup you used before.

In [32]:
def get_stage_optimizer_sgd(model, stage):
    momentum = 0.9
    weight_decay = 1e-4

    if stage == 1:
        lr = 1e-2
        print(f"Stage 1 LR (fc): {lr}")
        return optim.SGD(
            model.fc.parameters(),
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay
        )

    elif stage == 2:
        lr_backbone = 1e-3
        lr_head = 5e-3
        print(f"Stage 2 LR (layer4): {lr_backbone}, LR (fc): {lr_head}")
        return optim.SGD(
            [
                {"params": model.layer4.parameters(), "lr": lr_backbone},
                {"params": model.fc.parameters(), "lr": lr_head},
            ],
            momentum=momentum,
            weight_decay=weight_decay
        )

    elif stage == 3:
        lr = 1e-4
        print(f"Stage 3 LR (full model): {lr}")
        return optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay
        )

    else:
        raise ValueError("stage must be 1, 2, or 3")

# Learning Rate Scheduler

Reduce the learning rate during training to stabilize later epochs.

In [33]:
def get_scheduler(optimizer):
    return torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=3,
        gamma=0.1
    )

# Train for One Epoch

Train the model for one epoch on flipped training data and return average loss and accuracy.

In [34]:
def train_one_epoch(model, trainloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in trainloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs, dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    epoch_loss = running_loss / len(trainloader)
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc

Evaluate Clean and Superclass Accuracy

Evaluate the model on a dataloader and return average loss, Top-1 (fine) accuracy, and superclass (group) accuracy. This is used to measure both exact classification performance and coarse semantic correctness on clean validation and test sets.

In [35]:
def evaluate(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, preds = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc

def evaluate_superclass(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    group_correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, preds = torch.max(outputs, dim=1)

            total += labels.size(0)

            # compute superclass match
            for i in range(labels.size(0)):
                pred = preds[i].item()
                true = labels[i].item()

                if pred in LABEL_TO_GROUP[true]:
                    group_correct += 1

    epoch_loss = running_loss / len(dataloader)
    group_acc = 100.0 * group_correct / total

    return epoch_loss, group_acc

# Save Best Flipped Model During Training

Run one fine-tuning stage, track the best clean validation accuracy, and save the best checkpoint for the flipped model.

In [36]:
def run_stage(
    model,
    trainloader,
    valloader,
    criterion,
    optimizer,
    device,
    stage,
    epochs,
    scheduler=None,
    best_val_acc=0.0,
    best_model_path="best_flipped_model.pth"
):
    history = {
        "stage": stage,
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    print(f"\nStarting Stage {stage} for {epochs} epoch(s)\n")

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(
            model, trainloader, criterion, optimizer, device
        )

        val_loss, val_acc = evaluate(
            model, valloader, criterion, device
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"New best model saved with val acc: {val_acc:.2f}%")

        if scheduler is not None:
            scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Stage {stage} | "
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.2f}%"
        )

    return history, best_val_acc

# Load ResNet-18 Model

Load pretrained ResNet-18, replace the classifier for the selected dataset, and move the model to the correct device.

In [37]:
def build_model_for_dataset(dataset_name, device):
    num_classes = get_num_classes(dataset_name)

    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model = replace_resnet_classifier(model, num_classes=num_classes)
    model = model.to(device)

    return model

# Save Flipped Model Filename

Build a consistent filename for saving flipped models based on dataset and flip rate.

In [38]:
def get_flipped_model_path(dataset_name, flip_rate, save_dir="./saved_models"):
    filename = f"flipped_resnet18_{dataset_name}_fr{flip_rate:.2f}.pth"
    return os.path.join(save_dir, filename)

# Train One Flipped Model End-to-End

Fine-tune a flipped ResNet-18 using the three-stage schedule, save the best checkpoint, then evaluate clean test accuracy and super-class accuracy.

In [39]:
def train_flipped_model(
    dataset_name,
    flip_rate,
    dataloaders,
    stage_plan,
    device,
    save_dir="./saved_models"
):
    model = build_model_for_dataset(dataset_name, device)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    all_history = []

    best_model_path = get_flipped_model_path(
        dataset_name=dataset_name,
        flip_rate=flip_rate,
        save_dir=save_dir
    )

    for stage, epochs in stage_plan.items():
        model = set_finetune_stage(model, stage)
        optimizer = get_stage_optimizer_sgd(model, stage)
        scheduler = get_scheduler(optimizer)

        stage_history, best_val_acc = run_stage(
            model=model,
            trainloader=dataloaders["flipped_train"],
            valloader=dataloaders["clean_val"],
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            stage=stage,
            epochs=epochs,
            scheduler=scheduler,
            best_val_acc=best_val_acc,
            best_model_path=best_model_path
        )

        all_history.append(stage_history)

    best_model = build_model_for_dataset(dataset_name, device)
    best_model.load_state_dict(torch.load(best_model_path, map_location=device))
    best_model = best_model.to(device)

    clean_test_loss, clean_test_acc = evaluate(
        best_model, dataloaders["clean_test"], criterion, device
    )

    if dataset_name == 'cifar10':
      return {
          "model_path": best_model_path,
          "best_val_acc": best_val_acc,
          "clean_test_loss": clean_test_loss,
          "clean_test_acc": clean_test_acc,
          "history": all_history
      }

    superclass_test_loss, superclass_test_acc = evaluate_superclass(
        best_model, dataloaders["clean_test"], criterion, device
    )

    return {
        "model_path": best_model_path,
        "best_val_acc": best_val_acc,
        "clean_test_loss": clean_test_loss,
        "clean_test_acc": clean_test_acc,
        "superclass_test_loss": superclass_test_loss,
        "superclass_test_acc": superclass_test_acc,
        "history": all_history
    }

# Run One Flipped Training Example

Build dataloaders for one flip rate, train the flipped model, and print the resulting clean accuracy and superclass accuracy.

In [40]:
# example_dataset_name = "cifar10"
# example_flip_rate = 0.10

# example_datasets = build_datasets_for_flip_rate(
#     dataset_name=example_dataset_name,
#     flip_rate=example_flip_rate,
#     data_dir="./data",
#     val_ratio=VAL_RATIO,
#     img_size=IMG_SIZE,
#     seed=SEED
# )

# example_dataloaders = build_dataloaders(
#     example_datasets,
#     batch_size=BATCH_SIZE,
#     num_workers=NUM_WORKERS
# )

# example_result = train_flipped_model(
#     dataset_name=example_dataset_name,
#     flip_rate=example_flip_rate,
#     dataloaders=example_dataloaders,
#     stage_plan=STAGE_PLAN,
#     device=device,
#     save_dir=SAVE_DIR
# )

# print("Best Val Acc:", f"{example_result['best_val_acc']:.2f}%")
# print("Clean Test Acc:", f"{example_result['clean_test_acc']:.2f}%")
# if example_dataset_name == 'cifar100':
#   print(
#     f"Superclass Test Acc: {example_result['superclass_test_acc']:.2f}%"
#     if example_result['superclass_test_acc'] is not None
#     else "Superclass Test Acc: None"
#   )
# print("Saved model:", example_result["model_path"])

# Collect Prunable Parameters

Gather all convolution and linear layer weights that will be pruned.

In [41]:
def get_prunable_parameters(model):
    parameters_to_prune = []

    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            parameters_to_prune.append((module, "weight"))

    return parameters_to_prune

# Apply Global Unstructured Pruning

Apply global magnitude-based pruning across all convolution and linear weights in the model.

In [42]:
def apply_global_pruning(model, amount):
    parameters_to_prune = get_prunable_parameters(model)

    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=amount
    )

    return model

# Remove Pruning Reparameterization

Make pruning permanent by removing PyTorch pruning wrappers after masks have been applied.

In [43]:
def remove_pruning_reparam(model):
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            try:
                prune.remove(module, "weight")
            except ValueError:
                pass
    return model

# Measure Model Sparsity

Compute the percentage of zero weights after pruning across all prunable layers.

In [44]:
def measure_sparsity(model):
    zero_params = 0
    total_params = 0

    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            weight = module.weight.data
            zero_params += torch.sum(weight == 0).item()
            total_params += weight.numel()

    sparsity = 100.0 * zero_params / total_params
    return sparsity

# Pruned Model Optimizer

Define the optimizer used to recover performance after pruning.

In [45]:
def get_pruned_model_optimizer(model):
    return optim.SGD(
        model.parameters(),
        lr=1e-4,
        momentum=0.9,
        weight_decay=1e-4
    )

# Pruned Model Scheduler

Define the learning rate scheduler used during post-pruning recovery fine-tuning.

In [46]:
def get_pruned_model_scheduler(optimizer):
    return torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=3,
        gamma=0.1
    )

# Pruned Model Save Path

Build a consistent filename for each pruned model using dataset, flip rate, and sparsity level.

In [47]:
def get_pruned_model_path(dataset_name, flip_rate, prune_level, save_dir="./saved_models"):
    filename = (
        f"pruned_resnet18_{dataset_name}"
        f"_fr{flip_rate:.2f}"
        f"_sparse{prune_level:.2f}.pth"
    )
    return os.path.join(save_dir, filename)

# Fine-Tune One Pruned Model

Fine-tune a pruned model for recovery, save the best checkpoint based on clean validation accuracy, and return training history.

In [48]:
def finetune_pruned_model(
    model,
    trainloader,
    valloader,
    criterion,
    optimizer,
    device,
    epochs,
    scheduler=None,
    best_val_acc=0.0,
    best_model_path="best_pruned_model.pth"
):
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(
            model, trainloader, criterion, optimizer, device
        )

        val_loss, val_acc = evaluate(
            model, valloader, criterion, device
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"New best pruned model saved with val acc: {val_acc:.2f}%")

        if scheduler is not None:
            scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.2f}%"
        )

    return history, best_val_acc

# Load a Saved Flipped Model

Rebuild the correct ResNet-18 model for the selected dataset and load a saved flipped checkpoint.

In [49]:
def load_saved_flipped_model(dataset_name, model_path, device):
    model = build_model_for_dataset(dataset_name, device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    return model

# Run One Pruning Recovery Experiment

Load a flipped model, prune it to the chosen sparsity, recover it with fine-tuning, and evaluate clean accuracy and superclass accuracy.

In [50]:
def run_pruning_experiment(
    dataset_name,
    flip_rate,
    prune_level,
    flipped_model_path,
    dataloaders,
    device,
    recovery_epochs,
    save_dir="./saved_models"
):
    model = load_saved_flipped_model(dataset_name, flipped_model_path, device)
    criterion = nn.CrossEntropyLoss()

    model = apply_global_pruning(model, amount=prune_level)
    sparsity = measure_sparsity(model)
    model = remove_pruning_reparam(model)

    pruned_model_path = get_pruned_model_path(
        dataset_name=dataset_name,
        flip_rate=flip_rate,
        prune_level=prune_level,
        save_dir=save_dir
    )

    optimizer = get_pruned_model_optimizer(model)
    scheduler = get_pruned_model_scheduler(optimizer)

    history, best_val_acc = finetune_pruned_model(
        model=model,
        trainloader=dataloaders["flipped_train"],
        valloader=dataloaders["clean_val"],
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epochs=recovery_epochs,
        scheduler=scheduler,
        best_val_acc=0.0,
        best_model_path=pruned_model_path
    )

    best_model = load_saved_flipped_model(dataset_name, pruned_model_path, device)

    clean_test_loss, clean_test_acc = evaluate(
        best_model, dataloaders["clean_test"], criterion, device
    )

    if dataset_name == 'cifar10':
      return {
        "model_path": pruned_model_path,
        "best_val_acc": best_val_acc,
        "clean_test_loss": clean_test_loss,
        "clean_test_acc": clean_test_acc,
        "measured_sparsity": sparsity,
        "history": history
    }

    superclass_test_loss, superclass_test_acc = evaluate_superclass(
        best_model, dataloaders["clean_test"], criterion, device
    )

    return {
        "model_path": pruned_model_path,
        "best_val_acc": best_val_acc,
        "clean_test_loss": clean_test_loss,
        "clean_test_acc": clean_test_acc,
        "superclass_test_loss": superclass_test_loss,
        "superclass_test_acc": superclass_test_acc,
        "measured_sparsity": sparsity,
        "history": history
    }

# Run Full Flipping and Pruning Experiment

Iterate over all datasets, flip rates, and pruning levels. For each dataset and flip rate, train and save one flipped model, evaluate clean accuracy and superclass accuracy, then prune that flipped model at multiple sparsity levels, recover it with fine-tuning, evaluate again, and store all results.

In [78]:
# all_results = []

# for dataset_name in DATASETS:
#     for flip_rate in FLIP_RATES:
#         flipped_superclass_acc = None
#         pruned_superclass_acc = None

#         print("\n==========================================")
#         print(f"Dataset: {dataset_name} | Flip Rate: {flip_rate}")
#         print("==========================================\n")

#         # build datasets and dataloaders for this flip rate
#         datasets_dict = build_datasets_for_flip_rate(
#             dataset_name=dataset_name,
#             flip_rate=flip_rate,
#             data_dir="./data",
#             val_ratio=VAL_RATIO,
#             img_size=IMG_SIZE,
#             seed=SEED
#         )

#         dataloaders = build_dataloaders(
#             datasets_dict,
#             batch_size=BATCH_SIZE,
#             num_workers=NUM_WORKERS
#         )

#         # train flipped model
#         flipped_result = train_flipped_model(
#             dataset_name=dataset_name,
#             flip_rate=flip_rate,
#             dataloaders=dataloaders,
#             stage_plan=STAGE_PLAN,
#             device=device,
#             save_dir=SAVE_DIR
#         )

#         # compute clean accuracy drop for fliped model
#         baseline_acc = CLEAN_BASELINES[dataset_name]["test_acc"]
#         flipped_clean_acc_drop = baseline_acc - flipped_result["clean_test_acc"]

#         if dataset_name == 'cifar10':
#           flipped_experiment_result = {
#               "dataset": dataset_name,
#               "flip_rate": flip_rate,
#               "prune_level": 0.0,
#               "clean_test_acc": flipped_result["clean_test_acc"],
#               "clean_acc_drop": flipped_clean_acc_drop,
#               "best_val_acc": flipped_result["best_val_acc"],
#               "model_path": flipped_result["model_path"]
#           }
#         elif dataset_name == 'cifar100':
#           flipped_superclass_acc = flipped_result["superclass_test_acc"]

#           flipped_experiment_result = {
#               "dataset": dataset_name,
#               "flip_rate": flip_rate,
#               "prune_level": 0.0,
#               "clean_test_acc": flipped_result["clean_test_acc"],
#               "clean_acc_drop": flipped_clean_acc_drop,
#               "superclass_test_acc": flipped_superclass_acc,
#               "best_val_acc": flipped_result["best_val_acc"],
#               "model_path": flipped_result["model_path"]
#           }


#         all_results.append(flipped_experiment_result)

#         print("\nFlipped Model Result Summary:")
#         print(f"Clean Test Acc: {flipped_result['clean_test_acc']:.2f}%")
#         print(f"Clean Acc Drop: {flipped_clean_acc_drop:.2f}%")
#         print(
#             f"Superclass Acc: {flipped_superclass_acc:.2f}%"
#             if flipped_superclass_acc is not None
#             else "Superclass Acc: None"
#         )
#         print(f"Saved model: {flipped_result['model_path']}")

#         # prune the flipped model at all requested sparsity levels
#         for prune_level in PRUNE_LEVELS:

#             flipped_superclass_acc = None
#             pruned_superclass_acc = None

#             print("\n------------------------------------------")
#             print(f"Dataset: {dataset_name} | FLip Rate: {flip_rate} | Prune Level: {prune_level}")
#             print("------------------------------------------\n")

#             pruned_result = run_pruning_experiment(
#                 dataset_name=dataset_name,
#                 flip_rate=flip_rate,
#                 prune_level=prune_level,
#                 flipped_model_path=flipped_result["model_path"],
#                 dataloaders=dataloaders,
#                 device=device,
#                 recovery_epochs=PRUNED_RECOVERY_EPOCHS,
#                 save_dir=SAVE_DIR
#             )

#             pruned_clean_acc_drop = baseline_acc - pruned_result["clean_test_acc"]

#             if dataset_name == 'cifar10':

#               pruned_experiment_result = {
#                   "dataset": dataset_name,
#                   "flip_rate": flip_rate,
#                   "prune_level": prune_level,
#                   "clean_test_acc": pruned_result["clean_test_acc"],
#                   "clean_acc_drop": pruned_clean_acc_drop,
#                   "best_val_acc": pruned_result["best_val_acc"],
#                   "measured_sparsity": pruned_result["measured_sparsity"],
#                   "model_path": pruned_result["model_path"]
#               }

#             elif dataset_name == 'cifar100':
#               pruned_superclass_acc = pruned_result["superclass_test_acc"]

#               pruned_experiment_result = {
#                   "dataset": dataset_name,
#                   "flip_rate": flip_rate,
#                   "prune_level": prune_level,
#                   "clean_test_acc": pruned_result["clean_test_acc"],
#                   "clean_acc_drop": pruned_clean_acc_drop,
#                   "superclass_test_acc": pruned_superclass_acc,
#                   "best_val_acc": pruned_result["best_val_acc"],
#                   "measured_sparsity": pruned_result["measured_sparsity"],
#                   "model_path": pruned_result["model_path"]
#               }


#             all_results.append(pruned_experiment_result)

#             print("\nPruned Model Result Summary:")
#             print(f"Measured Sparsity: {pruned_result['measured_sparsity']:.2f}%")
#             print(f"Clean Test Acc: {pruned_result['clean_test_acc']:.2f}%")
#             print(f"Clean Acc Drop: {pruned_clean_acc_drop:.2f}%")
#             print(
#                 f"Superclass Acc: {pruned_superclass_acc:.2f}%"
#                 if pruned_superclass_acc is not None
#                 else "Superclass Acc: None"
#             )
#             print(f"Saved model: {pruned_result['model_path']}")


Dataset: cifar10 | Flip Rate: 0.1

Stage 1 LR (fc): 0.01

Starting Stage 1 for 2 epoch(s)

New best model saved with val acc: 76.20%
Stage 1 | Epoch [1/2] | Train Loss: 0.8069 | Train Acc: 72.22% | Val Loss: 0.6748 | Val Acc: 76.20%
New best model saved with val acc: 77.96%
Stage 1 | Epoch [2/2] | Train Loss: 0.6930 | Train Acc: 76.20% | Val Loss: 0.6272 | Val Acc: 77.96%
Stage 2 LR (layer4): 0.001, LR (fc): 0.005

Starting Stage 2 for 4 epoch(s)

New best model saved with val acc: 87.90%
Stage 2 | Epoch [1/4] | Train Loss: 0.4820 | Train Acc: 83.24% | Val Loss: 0.3490 | Val Acc: 87.90%
New best model saved with val acc: 89.66%
Stage 2 | Epoch [2/4] | Train Loss: 0.3614 | Train Acc: 87.18% | Val Loss: 0.3000 | Val Acc: 89.66%
New best model saved with val acc: 91.12%
Stage 2 | Epoch [3/4] | Train Loss: 0.3046 | Train Acc: 89.03% | Val Loss: 0.2663 | Val Acc: 91.12%
New best model saved with val acc: 91.40%
Stage 2 | Epoch [4/4] | Train Loss: 0.2494 | Train Acc: 91.13% | Val Loss: 0.25

100%|██████████| 169M/169M [00:13<00:00, 13.0MB/s]


Stage 1 LR (fc): 0.01

Starting Stage 1 for 2 epoch(s)

New best model saved with val acc: 50.78%
Stage 1 | Epoch [1/2] | Train Loss: 2.6105 | Train Acc: 37.29% | Val Loss: 1.8138 | Val Acc: 50.78%
New best model saved with val acc: 55.16%
Stage 1 | Epoch [2/2] | Train Loss: 2.0539 | Train Acc: 48.05% | Val Loss: 1.6314 | Val Acc: 55.16%
Stage 2 LR (layer4): 0.001, LR (fc): 0.005

Starting Stage 2 for 4 epoch(s)

New best model saved with val acc: 66.96%
Stage 2 | Epoch [1/4] | Train Loss: 1.6663 | Train Acc: 57.23% | Val Loss: 1.1283 | Val Acc: 66.96%
New best model saved with val acc: 69.08%
Stage 2 | Epoch [2/4] | Train Loss: 1.4446 | Train Acc: 62.54% | Val Loss: 1.0518 | Val Acc: 69.08%
New best model saved with val acc: 70.48%
Stage 2 | Epoch [3/4] | Train Loss: 1.3205 | Train Acc: 65.84% | Val Loss: 0.9952 | Val Acc: 70.48%
New best model saved with val acc: 71.72%
Stage 2 | Epoch [4/4] | Train Loss: 1.1784 | Train Acc: 69.89% | Val Loss: 0.9561 | Val Acc: 71.72%
Stage 3 LR (ful

# Save Full Experiment Results

Save all experiment results to a JSON file so they can be analyzed later without rerunning the notebook.

In [ ]:
# results_path = os.path.join(RESULTS_DIR, "flip_prune_results.json")

# with open(results_path, "w") as f:
#     json.dump(all_results, f, indent=4)

# print("Saved full experiment results to:", results_path)

# Preview Stored Results

Print a few example rows from the results list to verify the experiment outputs were recorded correctly.

In [80]:
# print("Total result rows:", len(all_results))

# for row in all_results[:5]:
#     print(row)

Total result rows: 30
{'dataset': 'cifar10', 'flip_rate': 0.1, 'prune_level': 0.0, 'clean_test_acc': 92.9, 'clean_acc_drop': 0.46999999999999886, 'best_val_acc': 93.42, 'model_path': './drive/MyDrive/DLBA Project/Label Flip Models/flipped_resnet18_cifar10_fr0.10.pth'}
{'dataset': 'cifar10', 'flip_rate': 0.1, 'prune_level': 0.2, 'clean_test_acc': 93.37, 'clean_acc_drop': 0.0, 'best_val_acc': 93.82, 'measured_sparsity': 19.999996419630737, 'model_path': './drive/MyDrive/DLBA Project/Label Flip Models/pruned_resnet18_cifar10_fr0.10_sparse0.20.pth'}
{'dataset': 'cifar10', 'flip_rate': 0.1, 'prune_level': 0.4, 'clean_test_acc': 93.41, 'clean_acc_drop': -0.03999999999999204, 'best_val_acc': 93.88, 'measured_sparsity': 40.00000179018463, 'model_path': './drive/MyDrive/DLBA Project/Label Flip Models/pruned_resnet18_cifar10_fr0.10_sparse0.40.pth'}
{'dataset': 'cifar10', 'flip_rate': 0.1, 'prune_level': 0.6, 'clean_test_acc': 93.07, 'clean_acc_drop': 0.30000000000001137, 'best_val_acc': 93.44, '

# Base Model Paths

Stores the paths to clean fine-tuned baseline models for CIFAR-10 and CIFAR-100.

In [56]:
BASE_MODEL_PATHS = {
    "cifar10": "/content/drive/MyDrive/DLBA Project/Models/finetuned_resnet18_cifar10_base_model.pth",
    "cifar100": "/content/drive/MyDrive/DLBA Project/Models/finetuned_resnet18_cifar100_base_model.pth",
}

# Load Saved Results JSON

Loads your existing flip/prune experiment results so you can re-evaluate each saved model.

In [58]:
import json
import os

results_path = os.path.join(RESULTS_DIR, "flip_prune_results.json")

with open(results_path, "r") as f:
    previous_results = json.load(f)

print("Loaded rows:", len(previous_results))

Loaded rows: 30


# Build Model From Saved Checkpoint

Loads a saved ResNet-18 checkpoint for CIFAR-10 or CIFAR-100.

In [59]:
def load_saved_model(dataset_name, model_path, device):
    model = build_model_for_dataset(dataset_name, device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    return model

# Evaluate CIFAR-10 Selected-Class Accuracy

Measures accuracy only on CIFAR-10 cat and dog samples.

In [60]:
def evaluate_selected_class_accuracy(model, dataloader, criterion, device, selected_labels):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    selected_labels = set(selected_labels)

    with torch.no_grad():
        for images, labels in dataloader:
            mask = torch.tensor([label.item() in selected_labels for label in labels], dtype=torch.bool)

            if mask.sum().item() == 0:
                continue

            images = images[mask].to(device)
            labels = labels[mask].to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, preds = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    avg_loss = running_loss / len(dataloader) if len(dataloader) > 0 else 0.0
    acc = 100.0 * correct / total if total > 0 else 0.0

    return avg_loss, acc

# Evaluate CIFAR-100 Superclass Accuracy

Uses your existing superclass evaluation function for CIFAR-100.

In [61]:
# already defined earlier:
# def evaluate_superclass(model, dataloader, criterion, device):
#     ...

# Build Clean Test Dataloaders for CIFAR-10 and CIFAR-100

Creates clean test loaders for both datasets so you can evaluate saved models consistently.

In [62]:
def build_clean_testloader_for_dataset(dataset_name):
    _, _, test_dataset = get_clean_datasets(
        dataset_name=dataset_name,
        data_dir="./data",
        val_ratio=VAL_RATIO,
        img_size=IMG_SIZE,
        seed=SEED
    )

    testloader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    )

    return testloader

clean_testloaders = {
    "cifar10": build_clean_testloader_for_dataset("cifar10"),
    "cifar100": build_clean_testloader_for_dataset("cifar100"),
}

100%|██████████| 170M/170M [00:13<00:00, 12.2MB/s]
100%|██████████| 169M/169M [00:13<00:00, 12.7MB/s]


# Evaluate Base Models on New Metrics

Computes the baseline selected-class accuracy for CIFAR-10 and baseline superclass accuracy for CIFAR-100.

In [63]:
criterion = nn.CrossEntropyLoss()

base_metric_results = {}

# CIFAR-10 base selected-class accuracy (cat/dog)
base_cifar10_model = load_saved_model("cifar10", BASE_MODEL_PATHS["cifar10"], device)
_, base_cifar10_selected_acc = evaluate_selected_class_accuracy(
    base_cifar10_model,
    clean_testloaders["cifar10"],
    criterion,
    device,
    selected_labels=[3, 5]
)

base_metric_results["cifar10"] = {
    "selected_class_acc": base_cifar10_selected_acc
}

# CIFAR-100 base superclass accuracy
base_cifar100_model = load_saved_model("cifar100", BASE_MODEL_PATHS["cifar100"], device)
_, base_cifar100_superclass_acc = evaluate_superclass(
    base_cifar100_model,
    clean_testloaders["cifar100"],
    criterion,
    device
)

base_metric_results["cifar100"] = {
    "superclass_acc": base_cifar100_superclass_acc
}

print(base_metric_results)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 198MB/s]


{'cifar10': {'selected_class_acc': 87.25}, 'cifar100': {'superclass_acc': 86.68}}


# Evaluate All Saved Models Against Base Metrics

Loads each saved model from your previous results and computes the new comparison metric and its drop from the clean base model.

In [64]:
metric_comparison_results = []

for row in previous_results:
    dataset_name = row["dataset"]
    model_path = row["model_path"]

    model = load_saved_model(dataset_name, model_path, device)

    result_row = dict(row)

    if dataset_name == "cifar10":
        _, selected_acc = evaluate_selected_class_accuracy(
            model,
            clean_testloaders["cifar10"],
            criterion,
            device,
            selected_labels=[3, 5]
        )

        base_selected_acc = base_metric_results["cifar10"]["selected_class_acc"]
        selected_acc_drop = base_selected_acc - selected_acc

        result_row["selected_class_acc"] = selected_acc
        result_row["selected_class_acc_drop"] = selected_acc_drop

    elif dataset_name == "cifar100":
        _, superclass_acc = evaluate_superclass(
            model,
            clean_testloaders["cifar100"],
            criterion,
            device
        )

        base_superclass_acc = base_metric_results["cifar100"]["superclass_acc"]
        superclass_acc_drop = base_superclass_acc - superclass_acc

        result_row["superclass_acc_recomputed"] = superclass_acc
        result_row["superclass_acc_drop"] = superclass_acc_drop

    metric_comparison_results.append(result_row)

print("Computed rows:", len(metric_comparison_results))

Computed rows: 30


# Save New Results File

Writes the updated results with selected-class or superclass-drop metrics into a new JSON file.

In [65]:
new_results_path = os.path.join(RESULTS_DIR, "flip_prune_results_with_group_metrics.json")

with open(new_results_path, "w") as f:
    json.dump(metric_comparison_results, f, indent=4)

print("Saved new results to:", new_results_path)

Saved new results to: ./drive/MyDrive/DLBA Project/Label Flip Results/flip_prune_results_with_group_metrics.json


# Preview New Results

Shows a few updated rows so you can verify the new fields were added correctly.

In [66]:
print("Total rows:", len(metric_comparison_results))

for row in metric_comparison_results[:5]:
    print(row)

Total rows: 30
{'dataset': 'cifar10', 'flip_rate': 0.1, 'prune_level': 0.0, 'clean_test_acc': 92.9, 'clean_acc_drop': 0.46999999999999886, 'best_val_acc': 93.42, 'model_path': './drive/MyDrive/DLBA Project/Label Flip Models/flipped_resnet18_cifar10_fr0.10.pth', 'selected_class_acc': 84.65, 'selected_class_acc_drop': 2.5999999999999943}
{'dataset': 'cifar10', 'flip_rate': 0.1, 'prune_level': 0.2, 'clean_test_acc': 93.37, 'clean_acc_drop': 0.0, 'best_val_acc': 93.82, 'measured_sparsity': 19.999996419630737, 'model_path': './drive/MyDrive/DLBA Project/Label Flip Models/pruned_resnet18_cifar10_fr0.10_sparse0.20.pth', 'selected_class_acc': 85.45, 'selected_class_acc_drop': 1.7999999999999972}
{'dataset': 'cifar10', 'flip_rate': 0.1, 'prune_level': 0.4, 'clean_test_acc': 93.41, 'clean_acc_drop': -0.03999999999999204, 'best_val_acc': 93.88, 'measured_sparsity': 40.00000179018463, 'model_path': './drive/MyDrive/DLBA Project/Label Flip Models/pruned_resnet18_cifar10_fr0.10_sparse0.40.pth', 'sel